# Demo Notebook: *On the representations of entities in Auto-Regressive Large Language Models*

This notebook demonstrates the experiments descibed in our papier *On the representations of entities in Auto-Regressive Large Language Models*



## Path 
please ensure that current working directory is the root of the repo

In [ ]:
import os 
wd = os.getcwd()
assert os.path.basename(wd) == "entityrepresentations", "Please run this script from the 'entityrepresentations' directory."

# Loadings

## Imports and Utils

In [ ]:
import torch, os, gc, sys
from pathlib import Path
from tqdm import tqdm
import numpy as np
from processResults import loadResults, get_taskVec
# read weights files
repo_path = Path(os.getcwd())
# load results
res_path = repo_path / "results"
if not res_path.exists():
    raise ValueError(
        "No results found, please make sure your current directory is the repository root"
    )

# get jobs
print(f"Loading results from {res_path}")
results = loadResults(res_path)

import transformer_lens as tl
from transformer_lens import HookedTransformer, patching

# our own code
import utils, plotting
from LabelExtractor import eval_model, infer_entities
import circuitsvis as cv
import plotly.io as pio

## Provided Results
We included the results for some of the conducted experiments in this repository. They are stored in the `results` folder. In each experiment we provide: 
- the Task Vector checkpoint `*.pth`
- the parameters used in the experiment `params.json` 
- the TV training history
- The output Evaluation of the TV on the test set

In this notebook, we have loaded them in a dataframe that enables us to quickly find and load task vectors for inference. 

In [ ]:
results.sort_values(by="layer").head(15).style.hide(
    subset=["hash", "path", "history", "Eval", "inference"], axis=1
    )  # hiding some columns for clarity

In [ ]:
# print unique model names
print(f" Results available for Models : {results['model_name'].unique()}")

## Params 
Thanks to the [`transformer_lens`](https://github.com/TransformerLensOrg/TransformerLens/tree/main) library, we can load and use many different LLMs seemlessly.

In [ ]:
# Load a model (eg GPT-2 Small)
model_name = "meta-llama/Meta-Llama-3-8B"  # ?
model_name = "gpt2-small"  # 117M ok
model_name = "pythia-2.8b"  # ok !
model_name = "gpt2-xl"  # 1.5B ok
model_name = "mistralai/Mistral-7B-v0.1"  # ok JZ a100 8cpus |
model_name = "gpt2-large"  # 774M ok
model_name = "gpt2-medium"  # 302M ok
model_name = "phi-1_5"  # 1.5B ok
model_name = "phi-2"  # 2,5B ok 12cpus nope, gpu 24cpus ok

with_context = True
with_context = False

## Load Model
By default, we try to load the model from the local hugginface cache, it this fails it will try to download it from HF. It may then need an additional acess token to download.

In [ ]:
# your huggingface token if you have one, needed for some models e;g Llama
hf_token = None

# check if model variable exists
if not "model" in locals():
    model = utils.load_llm(
        model_name,
        token=hf_token,
           dtype=torch.bfloat16
    )
    dim = model.QK.shape[-1]

model.eval()
model = model.cuda()
print(model)
print(f"Model {model_name} loaded as {model.W_U.dtype}")

# Entity Lens
Now that we have a LLM and a trained task vector $\theta_\ell$ loaded, we can infer entities from any representations.
We showcase here the *Entity Lens*, that generates a mention for each token considered at specifed layers, allowing to visualize to what *entity* the model is *thinking* in its internal representations.

### prompt

In [ ]:

prompt = "Somunkonwncitea is the capital of France, the capital of France is"
prompt = "When Albert Einstein and Mandelbrot went to the store, the General Relativity father gave the bottle to"
prompt = "Netherlands's capital is the city of"
prompt = "Gaston Julia and Mandelbrot meet, the latter tells"
prompt = "The City of Lights iconic landmark"
print("prompt:", prompt)

words = model.to_str_tokens(prompt)
print(len(words))
cv.tokens.colored_tokens(words, words)

### Inference

In [ ]:
from importlib import reload
reload(plotting)  # in case you edit the plotting file
from plotting import EntityLens

In [ ]:
from plotting import EntityLens

prompt = "The City of Lights iconic landmark"

every_N = 5
layers = [-1] + list(range(0, model.cfg.n_layers, every_N))
layers = [-1, 5, 10, 20, 25, 31]

data = EntityLens(
    model, results,
    layers=layers,
    prompt=prompt,
    with_context=with_context,
    use_TV_layer=20,  # 20
    # use_TV_layer=None,  # 20
    verbose=True,
    # compute_logit_lens=True,
    output="html",  # "fancy", "markdown", "html"
    plot_size=(740, 3500),  # (600, 5000) for the notebook
)

# Further analysis of the results

## Dataset

In [ ]:
dataset_name = "CoNLL2003"
train_dataset, test_dataset, val_dataset = utils.load_datasets(
    dataset_name, max_ent_length=200
)

In [ ]:
print("train length:", len(train_dataset))
print("dev length:", len(val_dataset))
print("test length:", len(test_dataset))
print("ex sample:")
item = train_dataset[np.random.randint(len(val_dataset))]
for key in item.keys():
    print(f" - {key}: {item[key]}")

In [ ]:
ind = np.random.randint(len(test_dataset))
# ind = 7616
prompt = test_dataset[ind]["text"]
prompt = test_dataset[np.random.randint(len(test_dataset))]["text"]
prompt = "Somunkonwncitea is the capital of France, the capital of France is"
prompt = "When Albert Einstein and Mandelbrot went to the store, the General Relativity father gave the bottle to"
prompt = "Netherlands's capital is the city of"
prompt = "Gaston Julia and Mandelbrot meet, the latter tells"
prompt = "The City of Lights iconic landmark"
print("prompt:", prompt)


words = model.to_str_tokens(prompt)
print(len(words))
cv.tokens.colored_tokens(words, words)

## Test Task Vectors

In [ ]:
layer = 20  # Layer at which we want to extract the trained task vector

fileName = get_taskVec(
    results,
    model_name,
    layer=layer,
    dataset_name=dataset_name,
    with_context=with_context,
)

TaskVec = torch.load(fileName, weights_only=True)
print("TaskVec loaded from ", fileName)

In [ ]:
# compute whole cache
print("computing cache ...")
# get whole hidden states
_, cache = model.run_with_cache(prompt)
repr = cache[tl.utils.get_act_name("resid_post", layer, "")][
    0, :, :
]  # 1 x n_tokens x dim
repr = repr.detach().cuda()
TaskVec = TaskVec.detach().to(repr.dtype).cuda()
print(repr.shape, repr.dtype)
data = [
    {
        "id": i,
        "text": prompt,
        "representation": repr[i],
        "tok": words[i],
    }
    for i in range(len(words))
]
infer_entities(model, TaskVec, data, with_context=with_context)

### Inference at given layer

In [ ]:
print(f"Prompt : '{data[0]['text']}' \n")
print("Token".center(20), f"| Inferred at layer {layer}")
print("-" * 60)
for it in data:
    print(f"{it['tok'].center(20)} | {it['inferred']}")

# smoother display in notebook
html = cv.tokens.colored_tokens(words, [it["inferred"] for it in data])
html.cdn_src = html.cdn_src.replace("margin: 15px", "margin: 50px")
html

## Interactive visualization

In [ ]:
from IPython.display import display
import ipywidgets as widgets

def interactive_plot():
    # nonlocal with_context, prompt, words, buttons, model, TaskVec
    # Callback function to update selected word
    def on_button_click(b):
        layer = dropdown.value
        repr = utils.get_representation(
            model,
            layer=layer,
            tokens=model.to_tokens(prompt),
            token_inds=torch.tensor([b.id]),
            verbose=True,
        )
        # change button color
        b.style.button_color = "lightgreen"
        # change all others to default
        for button in buttons:
            if button.id != b.id:
                button.style.button_color = "white"
        data = [{"id": 0, "representation": repr, "text": prompt}]
        infer_entities(model, TaskVec, data, with_context=with_context)
        selected_word_label.value = (
            f"Token: '{b.description}' \n Inferred: '{data[0]['inferred']}'"
        )
        # print("generation:", data[0]["inferred"])


    # Callback function to update prompt
    def on_text_submit(change):
        global prompt, words, buttons, button_box
        prompt = change["new"]
        words = model.to_str_tokens(prompt)
        print(words)
        print(len(words))

        # Update buttons
        buttons = []
        for ind, token in enumerate(words):
            buttons.append(
                clickableToken(
                    id=ind,
                    description=token,
                    layout=widgets.Layout(width="auto", margin="2px", padding="0 5px"),
                )
            )
        button_box.children = buttons


    # Textbox widget for entering prompt
    textbox = widgets.Text(
        value=prompt,
        description="Enter prompt:",
        disabled=False,
        layout=widgets.Layout(width="100%"),
    )
    textbox.observe(on_text_submit, names="value")

    # Buttons for each word
    buttons = []


    class clickableToken(widgets.Button):
        def __init__(self, id: int, description: str, **kwargs):
            super().__init__(description=description, **kwargs)
            self.description = description
            self.id = id
            self.on_click(on_button_click)


    for ind, token in enumerate(words):
        buttons.append(
            clickableToken(
                id=ind,
                description=token,
                layout=widgets.Layout(width="auto", margin="2px", padding="0 5px"),
            )
        )

    # Box widget with flexible wrapping
    button_box = widgets.Box(
        children=buttons,
        layout=widgets.Layout(display="flex", flex_flow="row wrap", align_items="center"),
    )
    # Dropdown widget for selecting an integer
    dropdown = widgets.Dropdown(
        options=[(f"layer {i}", i) for i in range(len(model.blocks) - 1, -1, -1)],
        description="Select a layer:",
        disabled=False,
    )
    dropdown.value = layer


    def on_checkbox_change(change):
        global with_context
        with_context = change["new"]


    # Create the checkbox widget
    with_context_checkbox = widgets.Checkbox(
        value=with_context, description="Generate with context", disabled=False
    )
    # Link the function to the checkbox change event
    with_context_checkbox.observe(on_checkbox_change, names="value")

    # Label to display the selected word
    selected_word_label = widgets.Label()

    # Create a title using HTML widget
    title = widgets.HTML(value="<h3>Select a Token to generate from:</h3>")

    # Display widgets
    display(textbox)
    display(title)
    display(button_box)
    # display checkbox and dropdown side by side
    display(widgets.HBox([with_context_checkbox, dropdown]))
    display(selected_word_label)


In [ ]:
interactive_plot()